# Silver Phase Notebook (Fresh)
SQL-first Bronze -> Silver workflow with Spark + Iceberg + Nessie.


In [26]:
# Step 1.1: Config
NESSIE_URL = "http://nessie:19120/api/v1"
BASE_BRANCH = "main"
TARGET_BRANCH = "dev"
SPARK_MASTER = "spark://spark:7077"
WAREHOUSE = "s3a://football-lake/"


In [30]:
# Step 1.2: Ensure Nessie branch exists (idempotent)
import requests
refs = requests.get(f"{NESSIE_URL}/trees", timeout=30).json().get("references", [])
main_ref = next((r for r in refs if r.get("type") == "BRANCH" and r.get("name") == BASE_BRANCH), None)
if main_ref is None:
    raise RuntimeError(f"Base branch {BASE_BRANCH!r} not found in Nessie.")
target_ref = next((r for r in refs if r.get("type") == "BRANCH" and r.get("name") == TARGET_BRANCH), None)
if target_ref is None:
    payload = {"type": "BRANCH", "name": TARGET_BRANCH, "hash": main_ref["hash"]}
    r = requests.post(f"{NESSIE_URL}/trees/tree", params={"sourceRefName": BASE_BRANCH}, json=payload, timeout=30)
    if r.status_code not in (200, 201, 409):
        raise RuntimeError(f"Failed creating branch: {r.status_code} {r.text}")
refs_after = requests.get(f"{NESSIE_URL}/trees", timeout=30).json().get("references", [])
main_after = next(r for r in refs_after if r.get("type") == "BRANCH" and r.get("name") == BASE_BRANCH)
target_after = next(r for r in refs_after if r.get("type") == "BRANCH" and r.get("name") == TARGET_BRANCH)
print("main hash  :", main_after["hash"])
print("target hash:", target_after["hash"])
print("branch_ready:", target_after["name"] == TARGET_BRANCH)


main hash  : 87b8bde39936e236a13d0fa43785b206d76d50f6e3d922d15c9cb6e9f6acd78b
target hash: 87b8bde39936e236a13d0fa43785b206d76d50f6e3d922d15c9cb6e9f6acd78b
branch_ready: True


In [31]:
# Step 1.3: Spark init (robust jar path detection)
import glob, os, sys
from pyspark.sql import SparkSession
for p in ["/opt/spark/python", "/usr/local/spark/python"]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)
jar_dirs = ["/opt/spark/jars-extra", "/usr/local/spark/jars-extra"]
jar_dir = next((d for d in jar_dirs if os.path.isdir(d)), None)
if jar_dir is None:
    raise RuntimeError(f"No jars-extra directory found. Checked: {jar_dirs}")
jars = sorted(glob.glob(f"{jar_dir}/*.jar"))
if not jars:
    raise RuntimeError(f"No jars found in {jar_dir}. Check docker-compose volume mount.")
try:
    spark.stop()
except Exception:
    pass
spark = (SparkSession.builder
    .appName("silver_sql_notebook_fresh")
    .master(SPARK_MASTER)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions")
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .config("spark.sql.catalog.nessie.uri", NESSIE_URL)
    .config("spark.sql.catalog.nessie.ref", TARGET_BRANCH)
    .config("spark.sql.catalog.nessie.warehouse", WAREHOUSE)
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true")
    .config("spark.sql.catalog.nessie.s3.region", "us-east-1")
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin")
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password")
    .config("spark.sql.defaultCatalog", "nessie")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.jars", ",".join(jars))
    .config("spark.driver.extraClassPath", f"{jar_dir}/*")
    .config("spark.executor.extraClassPath", f"{jar_dir}/*")
    .getOrCreate())
print("spark_catalog_ref:", spark.conf.get("spark.sql.catalog.nessie.ref"))
print("jar_dir:", jar_dir)
print("jar_count:", len(jars))


spark_catalog_ref: dev
jar_dir: /opt/spark/jars-extra
jar_count: 5


In [32]:
# Step 1.4: Validation
spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)
spark.sql("SHOW TABLES IN nessie.bronze").show(truncate=False)


26/04/27 19:31:04 WARN S3FileIO: Unclosed S3FileIO instance created by:
	org.apache.iceberg.aws.s3.S3FileIO.initialize(S3FileIO.java:359)
	org.apache.iceberg.CatalogUtil.loadFileIO(CatalogUtil.java:350)
	org.apache.iceberg.nessie.NessieCatalog.initialize(NessieCatalog.java:132)
	org.apache.iceberg.CatalogUtil.loadCatalog(CatalogUtil.java:255)
	org.apache.iceberg.CatalogUtil.buildIcebergCatalog(CatalogUtil.java:309)
	org.apache.iceberg.spark.SparkCatalog.buildIcebergCatalog(SparkCatalog.java:154)
	org.apache.iceberg.spark.SparkCatalog.initialize(SparkCatalog.java:751)
	org.apache.spark.sql.connector.catalog.Catalogs$.load(Catalogs.scala:65)
	org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$catalog$1(CatalogManager.scala:53)
	scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:86)
	org.apache.spark.sql.connector.catalog.CatalogManager.catalog(CatalogManager.scala:53)
	org.apache.spark.sql.connector.catalog.LookupCatalog$CatalogAndNamespace$.unapply(LookupCatalog.

+---------+
|namespace|
+---------+
|bronze   |
+---------+

+---------+---------------------------+-----------+
|namespace|tableName                  |isTemporary|
+---------+---------------------------+-----------+
|bronze   |events_raw                 |false      |
|bronze   |matches_raw                |false      |
|bronze   |player_team_match_stats_raw|false      |
+---------+---------------------------+-----------+



## Next (Silver SQL)
After validation passes, add SQL cells to create typed Silver tables from `nessie.bronze.player_team_match_stats_raw`.


In [53]:
spark.conf.set("spark.sql.shuffle.partitions", 64)

bronze_player_df = spark.table("nessie.bronze.player_team_match_stats_raw")
bronze_player_df.describe().show()
bronze_player_df.printSchema()
bronze_player_df.show(5)


+-------+------------------+-----------------+---------------+------------------+----------+------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+-------+------------------+------+--------------------+---------------+--------------------+-------------------+
|summary|          match_id|       player__id|   player__name|          team__id|team__name|       event_count|        pass_count|        shot_count|         goal_count|       assist_count| shot_assist_count|                 xg|started|        match_week|season|competition_id_input|season_id_input|     ingested_at_utc|             source|
+-------+------------------+-----------------+---------------+------------------+----------+------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+-------+------------------+------+--------------------+---------------+-------------------

## Define Silver Contract First

### 1- Canonical schema

In [42]:
from pyspark.sql import functions as F

silver_base_df = (
    bronze_player_df
    .select(
        F.col("match_id").alias("match_id"),
        F.col("player__id").alias("player_id"),
        F.col("player__name").alias("player_name"),
        F.col("team__id").alias("team_id"),
        F.col("team__name").alias("team_name"),
        F.col("season_id_input").alias("season_id"),
        F.col("match_week").alias("match_week"),
        F.col("competition_id_input").alias("competition_id"),
        F.col("source").alias("source"),
        F.col("event_count").alias("event_count"),
        F.col("pass_count").alias("pass_count"),
        F.col("shot_count").alias("shot_count"),
        F.col("goal_count").alias("goal_count"),
        F.col("assist_count").alias("assist_count"),
        F.col("shot_assist_count").alias("shot_assist_count"),
        F.col("xg").alias("xg"),
        F.col("ingested_at_utc").alias("ingested_at_utc")
    )
)

silver_typed_df = (
    silver_base_df
    .withColumn("match_id", F.col("match_id").cast("int"))
    .withColumn("player_id", F.col("player_id").cast("int"))
    .withColumn("team_id", F.col("team_id").cast("int"))
    .withColumn("season_id", F.col("season_id").cast("int"))
    .withColumn("match_week", F.col("match_week").cast("int"))
    .withColumn("competition_id", F.col("competition_id").cast("int"))
    .withColumn("event_count", F.col("event_count").cast("int"))
    .withColumn("pass_count", F.col("pass_count").cast("int"))
    .withColumn("shot_count", F.col("shot_count").cast("int"))
    .withColumn("goal_count", F.col("goal_count").cast("int"))
    .withColumn("assist_count", F.col("assist_count").cast("int"))
    .withColumn("shot_assist_count", F.col("shot_assist_count").cast("int"))
    .withColumn("xg", F.col("xg").cast("double"))
    .withColumn("ingested_at_utc", F.to_timestamp("ingested_at_utc"))
)

canonical_cols = [
    "match_id", "player_id", "team_id", "season_id", "match_week",                 # keys
    "player_name", "team_name", "competition_id", "source",                        # dimensions
    "event_count", "pass_count", "shot_count", "goal_count",                       # metrics
    "assist_count", "shot_assist_count", "xg",
    "ingested_at_utc"                                                               # metadata
]

silver_typed_df = silver_typed_df.select(*canonical_cols)

silver_typed_df.printSchema()
silver_typed_df.describe().show()
silver_typed_df.show(5, truncate=False)

root
 |-- match_id: integer (nullable = true)
 |-- player_id: integer (nullable = true)
 |-- team_id: integer (nullable = true)
 |-- season_id: integer (nullable = true)
 |-- match_week: integer (nullable = true)
 |-- player_name: string (nullable = true)
 |-- team_name: string (nullable = true)
 |-- competition_id: integer (nullable = true)
 |-- source: string (nullable = true)
 |-- event_count: integer (nullable = true)
 |-- pass_count: integer (nullable = true)
 |-- shot_count: integer (nullable = true)
 |-- goal_count: integer (nullable = true)
 |-- assist_count: integer (nullable = true)
 |-- shot_assist_count: integer (nullable = true)
 |-- xg: double (nullable = true)
 |-- ingested_at_utc: timestamp (nullable = true)



+-------+------------------+-----------------+------------------+---------+------------------+---------------+---------+--------------+-------------------+------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+
|summary|          match_id|        player_id|           team_id|season_id|        match_week|    player_name|team_name|competition_id|             source|       event_count|        pass_count|        shot_count|         goal_count|       assist_count| shot_assist_count|                 xg|
+-------+------------------+-----------------+------------------+---------+------------------+---------------+---------+--------------+-------------------+------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+
|  count|               306|              306|               306|      306|               306|            306|      306|    

### 2-Data quality contract

In [43]:
silver_clean_df = silver_typed_df.filter(F.col("match_id").isNotNull())
silver_clean_df = silver_clean_df.filter(F.col("player_id").isNotNull())
silver_clean_df = silver_clean_df.filter(F.col("team_id").isNotNull())
silver_clean_df = silver_clean_df.filter(F.col("season_id").isNotNull())
silver_clean_df = silver_clean_df.filter(F.col("match_week").isNotNull())

silver_clean_df = silver_clean_df.filter(F.col("xg") >= 0)

silver_clean_df.describe().show()
silver_clean_df.printSchema()
silver_clean_df.show(5, truncate=False)


+-------+------------------+-----------------+------------------+---------+------------------+---------------+---------+--------------+-------------------+------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+
|summary|          match_id|        player_id|           team_id|season_id|        match_week|    player_name|team_name|competition_id|             source|       event_count|        pass_count|        shot_count|         goal_count|       assist_count| shot_assist_count|                 xg|
+-------+------------------+-----------------+------------------+---------+------------------+---------------+---------+--------------+-------------------+------------------+------------------+------------------+-------------------+-------------------+------------------+-------------------+
|  count|               306|              306|               306|      306|               306|            306|      306|    

In [ ]:
# Hard checks
valid_count = silver_clean_df.count()
if valid_count == 0:
    raise RuntimeError("Hard check failed: no valid rows.")
null_ratio = silver_clean_df.filter(
    F.col("match_id").isNull() | F.col("player_id").isNull() | F.col("team_id").isNull()
).count() / valid_count
if null_ratio > 0.01:
    raise RuntimeError(f"Hard check failed: key null ratio too high ({null_ratio:.2%}).")

# Soft checks (example numbers)
xg_stats = silver_clean_df.agg(F.mean("xg").alias("xg_mean")).first()
dup_count = (silver_clean_df
    .groupBy("match_id", "player_id", "team_id")
    .count()
    .filter("count > 1")
    .count()
)
print(f"[SOFT] xg_mean={xg_stats['xg_mean']}, duplicate_keys={dup_count}")

[SOFT] xg_mean=0.10562219986764704, duplicate_keys=0


###  Dedup + Idempotency Strategy

In [46]:
# 1) Create namespace/table once (if not exists)
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")


spark.sql("""
CREATE TABLE IF NOT EXISTS nessie.silver.player_team_match_stats (
  match_id INT,
  player_id INT,
  player_name STRING,
  team_id INT,
  team_name STRING,
  season_id INT,
  match_week INT,
  competition_id INT,
  source STRING,
  event_count INT,
  pass_count INT,
  shot_count INT,
  goal_count INT,
  assist_count INT,
  shot_assist_count INT,
  xg DOUBLE,
  ingested_at_utc TIMESTAMP
)
USING iceberg
PARTITIONED BY (season_id, match_week)
""")

# 2) Optional: limit to one rerun scope (single week)
# silver_scope_df = silver_valid_df.filter((F.col("season_id")==281) & (F.col("match_week")==14))
silver_scope_df = silver_clean_df

# 3) Idempotent write for touched partitions
silver_scope_df.writeTo("nessie.silver.player_team_match_stats").overwritePartitions()

In [ ]:
## validate 
spark.sql("""
SELECT season_id, match_week, COUNT(*) AS rows
FROM nessie.silver.player_team_match_stats
GROUP BY season_id, match_week
ORDER BY season_id, match_week
""").show(100, truncate=False)

+---------+----------+----+
|season_id|match_week|rows|
+---------+----------+----+
|281      |5         |30  |
|281      |8         |29  |
|281      |10        |31  |
|281      |11        |31  |
|281      |13        |30  |
|281      |14        |30  |
|281      |15        |30  |
|281      |18        |32  |
|281      |19        |31  |
|281      |20        |32  |
+---------+----------+----+



- snapshot retention policy set to avoid saving old unneeded snapshots 
- distribution-mode set to be hash (data is shuffled using the partition key (hashcode))
- and file size set to primarily prevents files from becoming too large

In [ ]:
spark.sql("""
ALTER TABLE nessie.silver.player_team_match_stats
SET TBLPROPERTIES (
  
  -- write behavior
  'write.distribution-mode'='hash',
  'write.target-file-size-bytes'='268435456',   -- 256 MB
  
  -- metadata/snapshot hygiene
  'history.expire.max-snapshot-age-ms'='604800000',  -- 7 days
  'history.expire.min-snapshots-to-keep'='5',
  
  -- delete file handling
  'write.delete.mode'='merge-on-read',
  'write.update.mode'='merge-on-read',
  'write.merge.mode'='merge-on-read'
);
""")

DataFrame[]

In [52]:
spark.sql("""
SHOW TBLPROPERTIES nessie.silver.player_team_match_stats;
""").show()


+--------------------+--------------------+
|                 key|               value|
+--------------------+--------------------+
| current-snapshot-id| 6262345497204398465|
|              format|     iceberg/parquet|
|      format-version|                   2|
|          gc.enabled|               false|
|history.expire.ma...|           604800000|
|    nessie.commit.id|16befa0a01101b618...|
|write.distributio...|                hash|
|write.metadata.de...|               false|
|write.parquet.com...|                zstd|
|write.target-file...|           268435456|
+--------------------+--------------------+



### compaction small files with validation before and after

In [ ]:
spark.sql("""
SELECT
  count(*) AS file_count,
  round(avg(file_size_in_bytes)/1024/1024, 2) AS avg_size_mb,
  round(sum(file_size_in_bytes)/1024/1024, 2) AS total_size_mb
FROM nessie.silver.player_team_match_stats.files
WHERE partition.season_id = 281 AND partition.match_week = 14
""").show(truncate=False)


spark.sql("""
CALL nessie.system.rewrite_data_files(
  table => 'silver.player_team_match_stats',# after metrics

  where => 'season_id = 281 AND match_week = 14'
)
""").show(truncate=False)

spark.sql("""
SELECT
  count(*) AS file_count,
  round(avg(file_size_in_bytes)/1024/1024, 2) AS avg_size_mb,
  round(sum(file_size_in_bytes)/1024/1024, 2) AS total_size_mb
FROM nessie.silver.player_team_match_stats.files
WHERE partition.season_id = 281 AND partition.match_week = 14
""").show(truncate=False)

+----------+-----------+-------------+
|file_count|avg_size_mb|total_size_mb|
+----------+-----------+-------------+
|1         |0.01       |0.01         |
+----------+-----------+-------------+

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|0                         |0                     |0                    |0                      |
+--------------------------+----------------------+---------------------+-----------------------+

+----------+-----------+-------------+
|file_count|avg_size_mb|total_size_mb|
+----------+-----------+-------------+
|1         |0.01       |0.01         |
+----------+-----------+-------------+

